# RF + ECFP with Threshold Tuning & DAG Consistency

Building on rf_ecfp baseline (Val F1: 0.4804), this notebook adds:
1. Per-class threshold optimization
2. DAG consistency enforcement (parent prob >= child prob)

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
from skfp.fingerprints import ECFPFingerprint
from skfp.preprocessing import MolFromSmilesTransformer
from collections import defaultdict
import re

# Load data
train_df = pd.read_parquet('../data/train.parquet')
val_df = pd.read_parquet('../data/val.parquet')
test_df = pd.read_parquet('../chebi_dataset_test_empty.parquet')
label_cols = [c for c in train_df.columns if c.startswith('class_')]

print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}, Classes: {len(label_cols)}")

Train: 26934, Val: 6734, Test: 11223, Classes: 500


In [2]:
# Compute ECFP fingerprints
mol_transformer = MolFromSmilesTransformer()
ecfp = ECFPFingerprint(fp_size=2048, radius=2, count=True)

X_train = ecfp.transform(mol_transformer.transform(train_df['SMILES'].tolist()))
X_val = ecfp.transform(mol_transformer.transform(val_df['SMILES'].tolist()))
X_test = ecfp.transform(mol_transformer.transform(test_df['SMILES'].tolist()))

y_train = train_df[label_cols].values
y_val = val_df[label_cols].values

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}, X_test: {X_test.shape}")

[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not removing hydrogen atom without neighbors
[14:16:59] WARNING: not r

X_train: (26934, 2048), X_val: (6734, 2048), X_test: (11223, 2048)


In [3]:
# Train RF and get probability predictions
rf = RandomForestClassifier(n_estimators=50, n_jobs=-1, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

# For multi-output RF, predict_proba returns list of arrays (one per class)
proba_list = rf.predict_proba(X_val)
y_val_proba = np.array([p[:, 1] if p.shape[1] == 2 else p[:, 0] for p in proba_list]).T

print(f"Val proba shape: {y_val_proba.shape}, range: [{y_val_proba.min():.3f}, {y_val_proba.max():.3f}]")

Val proba shape: (6734, 500), range: [0.000, 1.000]


In [4]:
# Baseline F1 with default threshold
y_val_pred_default = (y_val_proba >= 0.5).astype(int)
f1_default = np.mean([f1_score(y_val[:, i], y_val_pred_default[:, i], zero_division=0) for i in range(len(label_cols))])
print(f"Baseline Val Macro F1 (threshold=0.5): {f1_default:.4f}")

Baseline Val Macro F1 (threshold=0.5): 0.4809


## Part 1: Per-class Threshold Optimization

In [5]:
def optimize_threshold(y_true, y_proba, thresholds=np.arange(0.1, 0.91, 0.05)):
    """Find optimal threshold for a single class."""
    best_thresh = 0.5
    best_f1 = 0
    for t in thresholds:
        y_pred = (y_proba >= t).astype(int)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_thresh = t
    return best_thresh, best_f1

# Optimize threshold for each class
optimal_thresholds = []
per_class_f1_optimized = []

for i in range(len(label_cols)):
    thresh, f1 = optimize_threshold(y_val[:, i], y_val_proba[:, i])
    optimal_thresholds.append(thresh)
    per_class_f1_optimized.append(f1)

optimal_thresholds = np.array(optimal_thresholds)
print(f"Threshold stats: mean={optimal_thresholds.mean():.3f}, min={optimal_thresholds.min():.3f}, max={optimal_thresholds.max():.3f}")
print(f"Optimized Val Macro F1: {np.mean(per_class_f1_optimized):.4f}")

Threshold stats: mean=0.309, min=0.100, max=0.900
Optimized Val Macro F1: 0.7012


## Part 2: Parse DAG from OBO file

In [6]:
def parse_obo(filepath):
    """Parse OBO file to extract parent-child relationships."""
    parents = defaultdict(list)  # child -> [parents]
    children = defaultdict(list)  # parent -> [children]
    
    current_id = None
    with open(filepath, 'r') as f:
        for line in f:
            line = line.strip()
            if line.startswith('id: class_'):
                current_id = line.split('id: ')[1]  # e.g., "class_0"
            elif line.startswith('is_a: class_') and current_id:
                parent_id = line.split('is_a: ')[1].split(' !')[0]  # e.g., "class_1"
                parents[current_id].append(parent_id)
                children[parent_id].append(current_id)
    
    return dict(parents), dict(children)

parents_map, children_map = parse_obo('../data/chebi_classes.obo')
print(f"Parsed {len(parents_map)} nodes with parents, {len(children_map)} nodes with children")

Parsed 499 nodes with parents, 312 nodes with children


In [7]:
# Map class columns to IDs - columns are already in format "class_X"
# So we can use them directly as keys
col_to_id = {col: col for col in label_cols}  # identity mapping
id_to_col = col_to_id.copy()
id_to_idx = {col: i for i, col in enumerate(label_cols)}

print(f"Mapped {len(id_to_idx)} columns")
print(f"Sample: {label_cols[:3]}")
print(f"Classes with parents in DAG: {len([c for c in label_cols if c in parents_map])}")

Mapped 500 columns
Sample: ['class_0', 'class_1', 'class_2']
Classes with parents in DAG: 499


In [8]:
def enforce_dag_consistency(proba, parents_map, id_to_idx):
    """
    Enforce DAG consistency: P(parent) >= max(P(children))
    Vectorized bottom-up propagation for efficiency.
    """
    proba = proba.copy()
    our_ids = set(id_to_idx.keys())
    
    # Iterate until no changes (propagate up the DAG)
    for _ in range(50):  # max iterations (DAG depth)
        changed = False
        for child_id, parent_ids in parents_map.items():
            if child_id not in our_ids:
                continue
            child_idx = id_to_idx[child_id]
            
            for parent_id in parent_ids:
                if parent_id not in our_ids:
                    continue
                parent_idx = id_to_idx[parent_id]
                
                # Vectorized: update all samples at once
                mask = proba[:, parent_idx] < proba[:, child_idx]
                if mask.any():
                    proba[mask, parent_idx] = proba[mask, child_idx]
                    changed = True
        
        if not changed:
            break
    
    return proba

# Apply DAG consistency to validation probabilities
print("Applying DAG consistency...")
y_val_proba_dag = enforce_dag_consistency(y_val_proba, parents_map, id_to_idx)
print(f"Done. Proba range: [{y_val_proba_dag.min():.3f}, {y_val_proba_dag.max():.3f}]")

Applying DAG consistency...
Done. Proba range: [0.000, 1.000]


## Part 3: Count DAG Inconsistencies

In [9]:
def count_inconsistencies(proba, parents_map, id_to_idx):
    """Count cases where child prob > parent prob."""
    our_ids = set(id_to_idx.keys())
    total = 0
    
    for sample_idx in range(len(proba)):
        for child_id, parent_ids in parents_map.items():
            if child_id not in our_ids:
                continue
            child_idx = id_to_idx[child_id]
            child_prob = proba[sample_idx, child_idx]
            
            for parent_id in parent_ids:
                if parent_id not in our_ids:
                    continue
                parent_idx = id_to_idx[parent_id]
                if proba[sample_idx, parent_idx] < child_prob:
                    total += 1
    
    return total

incons_before = count_inconsistencies(y_val_proba, parents_map, id_to_idx)
incons_after = count_inconsistencies(y_val_proba_dag, parents_map, id_to_idx)
print(f"DAG inconsistencies - Before: {incons_before}, After: {incons_after}")

# Re-optimize thresholds on DAG-consistent probabilities
optimal_thresholds_dag = []
per_class_f1_dag = []

for i in range(len(label_cols)):
    thresh, f1 = optimize_threshold(y_val[:, i], y_val_proba_dag[:, i])
    optimal_thresholds_dag.append(thresh)
    per_class_f1_dag.append(f1)

optimal_thresholds_dag = np.array(optimal_thresholds_dag)
print(f"DAG Threshold stats: mean={optimal_thresholds_dag.mean():.3f}, min={optimal_thresholds_dag.min():.3f}, max={optimal_thresholds_dag.max():.3f}")
print(f"DAG + Threshold Optimized Val Macro F1: {np.mean(per_class_f1_dag):.4f}")

DAG inconsistencies - Before: 0, After: 0
DAG Threshold stats: mean=0.309, min=0.100, max=0.900
DAG + Threshold Optimized Val Macro F1: 0.7012


## Part 4: Generate Test Predictions

In [10]:
# Retrain on full data (train + val)
full_df = pd.concat([train_df, val_df], ignore_index=True)
X_full = ecfp.transform(mol_transformer.transform(full_df['SMILES'].tolist()))
y_full = full_df[label_cols].values

rf_full = RandomForestClassifier(n_estimators=50, n_jobs=-1, class_weight='balanced', random_state=42)
rf_full.fit(X_full, y_full)
print("Trained on full data")

[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not removing hydrogen atom without neighbors
[14:17:46] WARNING: not r

Trained on full data


In [11]:
# Get test probabilities
proba_list_test = rf_full.predict_proba(X_test)
y_test_proba = np.array([p[:, 1] if p.shape[1] == 2 else p[:, 0] for p in proba_list_test]).T

print(f"Test proba shape: {y_test_proba.shape}")

Test proba shape: (11223, 500)


In [12]:
# Apply DAG consistency
y_test_proba_dag = enforce_dag_consistency(y_test_proba, parents_map, id_to_idx)

# Apply optimized thresholds
y_test_pred = np.zeros_like(y_test_proba_dag, dtype=int)
for i in range(len(label_cols)):
    y_test_pred[:, i] = (y_test_proba_dag[:, i] >= optimal_thresholds_dag[i]).astype(int)

print(f"Test predictions shape: {y_test_pred.shape}")
print(f"Avg predictions per sample: {y_test_pred.sum(axis=1).mean():.1f}")

Test predictions shape: (11223, 500)
Avg predictions per sample: 31.0


In [13]:
# Save submission
submission = pd.DataFrame(y_test_pred, columns=label_cols)
submission.insert(0, 'mol_id', test_df['mol_id'].values)
submission.insert(1, 'SMILES', test_df['SMILES'].values)
submission.to_parquet('../submissions/rf_ecfp_tuned.parquet', index=False)
print(f"Saved submission: {submission.shape}")

Saved submission: (11223, 502)


In [14]:
# Summary
print("="*50)
print("SUMMARY")
print("="*50)
print(f"Baseline (threshold=0.5):     Val F1 = {f1_default:.4f}")
print(f"+ Threshold tuning:           Val F1 = {np.mean(per_class_f1_optimized):.4f}")
print(f"+ DAG consistency + tuning:   Val F1 = {np.mean(per_class_f1_dag):.4f}")
print(f"DAG inconsistencies: {incons_before} -> {incons_after}")

SUMMARY
Baseline (threshold=0.5):     Val F1 = 0.4809
+ Threshold tuning:           Val F1 = 0.7012
+ DAG consistency + tuning:   Val F1 = 0.7012
DAG inconsistencies: 0 -> 0
